# TimeRAG ERA5 数据下载（Jupyter 教程版，独立实现）

目标：
- 查询城市（10 个）：1991-2023 年逐月 2m 温度
- 目标城市（汉中）：2024 年逐月 2m 温度

本教程所有下载逻辑均在 notebook 内实现，不依赖仓库脚本，可逐格执行。

## 1. 安装依赖
如环境已具备 `cdsapi`, `xarray`, `netCDF4`, `numpy`, `tqdm`, `matplotlib` 可跳过。

In [1]:
%pip install -q cdsapi xarray netCDF4 numpy tqdm matplotlib

Note: you may need to restart the kernel to use updated packages.


## 2. 配置 `.cdsapirc`
- 输入保存路径（默认 `/.cdsapirc`）。
- 输入 API key（不回显）。
文件内容示例：
```
url: https://cds.climate.copernicus.eu/api
key: <api-key>

API申请：https://cds.climate.copernicus.eu/how-to-api
(建议使用Google邮箱注册)
```

## 3. 定义城市与年份范围
- 查询城市（1991-2023）
- 目标城市（汉中，2024）

In [2]:
QUERY_CITIES = {
    '西安': {'lat': 34.27, 'lon': 108.95},
    '郑州': {'lat': 34.75, 'lon': 113.62},
    '太原': {'lat': 37.87, 'lon': 112.55},
    '石家庄': {'lat': 38.04, 'lon': 114.51},
    '济南': {'lat': 36.65, 'lon': 117.12},
    '呼和浩特': {'lat': 40.82, 'lon': 111.65},
    '银川': {'lat': 38.47, 'lon': 106.27},
    '兰州': {'lat': 36.06, 'lon': 103.84},
    '西宁': {'lat': 36.62, 'lon': 101.78},
    '大同': {'lat': 40.08, 'lon': 113.30},
}
TARGET_CITY = {'汉中': {'lat': 33.07, 'lon': 107.03}}
QUERY_YEARS = (1991, 2023)
TARGET_YEARS = (2024, 2024)

## 4. 下载工具

- 按月下载，便于断点续传。
- 对每个城市请求微小区域（±0.25°），减小文件体积。
- 每年保存两个 `.npy`：数据与时间戳。
- 进度写入 `download_progress.json`，防止重复下载。

In [3]:
import cdsapi
import numpy as np
import xarray as xr
import os, json, tempfile, time, threading
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

OUTPUT_DIR = Path("shaanxi_cities_temperature")  # 可改
OUTPUT_DIR.mkdir(exist_ok=True)
PROGRESS_FILE = OUTPUT_DIR / "download_progress.json"
progress_lock = threading.Lock()

# 读取/保存进度（线程安全）

def load_progress():
    with progress_lock:
        if PROGRESS_FILE.exists():
            return json.loads(PROGRESS_FILE.read_text())
        return {}

def save_progress(p):
    with progress_lock:
        PROGRESS_FILE.write_text(json.dumps(p, indent=2, ensure_ascii=False))

# 工具函数

def small_area(lat, lon, buffer=0.25):
    return [lat + buffer, lon - buffer, lat - buffer, lon + buffer]

def days_in_month(year, month):
    if month == 12:
        return 31
    return (datetime(year if month < 12 else year+1, (month % 12)+1, 1) - datetime(year, month, 1)).days

def download_month(city_name, lat, lon, year, month, client):
    days = [f"{d:02d}" for d in range(1, days_in_month(year, month) + 1)]
    area = small_area(lat, lon)
    tmp = tempfile.NamedTemporaryFile(suffix='.nc', delete=False)
    tmp_path = tmp.name
    tmp.close()
    try:
        client.retrieve(
            'reanalysis-era5-single-levels',
            {
                'product_type': 'reanalysis',
                'variable': '2m_temperature',
                'year': str(year),
                'month': f'{month:02d}',
                'day': days,
                'time': [f"{h:02d}:00" for h in range(24)],
                'area': area,
                'data_format': 'netcdf',
            },
            tmp_path
        )
        ds = xr.open_dataset(tmp_path)
        city_ds = ds.interp(latitude=lat, longitude=lon, method='linear')
        t2m = city_ds['t2m'].values
        times = (city_ds['time'] if 'time' in city_ds.coords else city_ds['valid_time']).values
        ds.close()
        os.remove(tmp_path)
        return t2m, times
    except Exception as e:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        raise e

def download_year(city_name, coords, year, client, max_retries=3):
    lat, lon = coords['lat'], coords['lon']
    data_all, time_all = [], []
    for month in range(1, 13):
        for attempt in range(1, max_retries+1):
            try:
                t2m, ts = download_month(city_name, lat, lon, year, month, client)
                data_all.extend(t2m)
                time_all.extend(ts)
                break
            except Exception as e:
                print(f"  {city_name} {year}-{month:02d} 尝试 {attempt}/{max_retries} 失败: {e}")
                if attempt == max_retries:
                    raise
                time.sleep(5)
    data_all = np.array(data_all)
    time_all = np.array(time_all)
    np.save(OUTPUT_DIR / f"{city_name}_{year}.npy", data_all)
    np.save(OUTPUT_DIR / f"{city_name}_{year}_times.npy", time_all)


def _download_one(city_name, coords, year):
    """单个 (city, year) 下载任务，线程池回调函数"""
    client = cdsapi.Client()  # 每个线程创建独立的 client
    download_year(city_name, coords, year, client)
    # 完成后更新进度
    progress = load_progress()
    progress[f"{city_name}_{year}"] = "done"
    save_progress(progress)
    return city_name, year


def download_range(cities, start_year, end_year, max_workers=2):
    """双线程并行下载"""
    progress = load_progress()
    tasks = [
        (city, coords, year)
        for city, coords in cities.items()
        for year in range(start_year, end_year + 1)
        if progress.get(f"{city}_{year}") != "done"
    ]
    if not tasks:
        print("全部已完成，无需下载。")
        return
    print(f"待下载任务数: {len(tasks)}，使用 {max_workers} 个线程并行下载")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_download_one, city, coords, year): (city, year)
            for city, coords, year in tasks
        }
        for future in as_completed(futures):
            city, year = futures[future]
            try:
                future.result()
                print(f"✅ {city} {year} 下载完成")
            except Exception as e:
                print(f"❌ {city} {year} 下载失败: {e}")
    print("全部完成。")


## 5. 下载查询城市（1991-2023）
首次运行时间较长，支持断点续传（基于 `download_progress.json`）。

In [4]:
download_range(QUERY_CITIES, QUERY_YEARS[0], QUERY_YEARS[1])

全部已完成，无需下载。


## 6. 下载目标城市汉中（2024）

In [5]:
download_range(TARGET_CITY, TARGET_YEARS[0], TARGET_YEARS[1])

全部已完成，无需下载。


## 7. 列出已下载文件

In [6]:
import glob
for f in sorted(glob.glob(str(OUTPUT_DIR / '*.npy'))):
    print(f)

shaanxi_cities_temperature\兰州_1991.npy
shaanxi_cities_temperature\兰州_1991_times.npy
shaanxi_cities_temperature\兰州_1992.npy
shaanxi_cities_temperature\兰州_1992_times.npy
shaanxi_cities_temperature\兰州_1993.npy
shaanxi_cities_temperature\兰州_1993_times.npy
shaanxi_cities_temperature\兰州_1994.npy
shaanxi_cities_temperature\兰州_1994_times.npy
shaanxi_cities_temperature\兰州_1995.npy
shaanxi_cities_temperature\兰州_1995_times.npy
shaanxi_cities_temperature\兰州_1996.npy
shaanxi_cities_temperature\兰州_1996_times.npy
shaanxi_cities_temperature\兰州_1997.npy
shaanxi_cities_temperature\兰州_1997_times.npy
shaanxi_cities_temperature\兰州_1998.npy
shaanxi_cities_temperature\兰州_1998_times.npy
shaanxi_cities_temperature\兰州_1999.npy
shaanxi_cities_temperature\兰州_1999_times.npy
shaanxi_cities_temperature\兰州_2000.npy
shaanxi_cities_temperature\兰州_2000_times.npy
shaanxi_cities_temperature\兰州_2001.npy
shaanxi_cities_temperature\兰州_2001_times.npy
shaanxi_cities_temperature\兰州_2002.npy
shaanxi_cities_temperature\兰州_2002_ti

## 8. 生成查询城市1991-2023年的合并文件

In [ ]:
import numpy as np
from pathlib import Path

# 数据路径
DATA_ROOT = Path('./shaanxi_cities_temperature')
QUERY_CITIES = ['西安','郑州','太原','石家庄','济南','呼和浩特','银川','兰州','西宁','大同']

for city in QUERY_CITIES:
    all_data = []
    all_times = []
    
    # 遍历 1991 到 2023 年
    for year in range(1991, 2024):
        data_file = DATA_ROOT / f"{city}_{year}.npy"
        time_file = DATA_ROOT / f"{city}_{year}_times.npy"
        
        if data_file.exists() and time_file.exists():
            all_data.append(np.load(data_file))
            all_times.append(np.load(time_file))
        else:
            print(f"⚠️ 缺失文件: {city} {year} 年")
            
    if all_data:
        # 按时间维度(axis=0)拼接数据
        merged_data = np.concatenate(all_data, axis=0)
        merged_times = np.concatenate(all_times, axis=0)
        
        # 写入最终合并的文件
        out_data_path = DATA_ROOT / f"{city}_1991-2023.npy"
        out_time_path = DATA_ROOT / f"{city}_1991-2023_times.npy"
        
        np.save(out_data_path, merged_data)
        np.save(out_time_path, merged_times)
        
        print(f"✅ 成功生成合并文件: {out_data_path.name} | 数据长度: {len(merged_data)}")


✅ 成功生成合并文件: 西安_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 郑州_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 太原_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 石家庄_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 济南_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 呼和浩特_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 银川_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 兰州_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 西宁_1991-2023.npy | 数据长度: 289272
✅ 成功生成合并文件: 大同_1991-2023.npy | 数据长度: 289272
